# Diagnose & Fix: Anna Asset Delivery Dates - Ghost Item

**Problem:** The Power BI Imports API returns `ReportWithDisplayNameForTheModelExists` (409) with `existingReport: null` when importing "Anna Asset Delivery Dates". This means a ghost/orphaned dataset or report is blocking the name, invisible to the Fabric Items API but present in the Power BI backend.

**This notebook will:**
1. Authenticate and discover workspace items
2. Search for ANY item named "Anna" across all API surfaces
3. Identify and delete the ghost item
4. (Optional) Re-upload the RDL to confirm the fix

## Cell 1 - Configuration
Set your workspace ID and report name below.

In [ ]:
# === CONFIGURATION ===
# Update these values for your environment
WORKSPACE_ID = "68c48a5c-8b88-4559-962d-5dc7f29ab96a"  # UAT workspace from the deployment log
REPORT_NAME = "Anna Asset Delivery Dates"

# API base URLs
FABRIC_API = "https://api.fabric.microsoft.com/v1"
PBI_API = "https://api.powerbi.com/v1.0/myorg"

print(f"Target workspace: {WORKSPACE_ID}")
print(f"Looking for: '{REPORT_NAME}'")

## Cell 2 - Authentication
Uses the Fabric notebook's built-in token (no credentials needed when running in a Fabric workspace).

In [ ]:
import requests
import json

# In a Fabric notebook, use the built-in credential
try:
    from notebookutils import mssparkutils
    access_token = mssparkutils.credentials.getToken("https://analysis.windows.net/powerbi/api")
    print("\u2713 Authenticated via Fabric notebook token")
except ImportError:
    # Fallback: if running locally, use azure-identity
    from azure.identity import DefaultAzureCredential
    credential = DefaultAzureCredential()
    token = credential.get_token("https://analysis.windows.net/powerbi/api/.default")
    access_token = token.token
    print("\u2713 Authenticated via DefaultAzureCredential")

headers = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json"
}

def api_get(url):
    """Helper for GET requests with error handling."""
    resp = requests.get(url, headers=headers)
    if resp.ok:
        return resp.json()
    else:
        print(f"  \u2717 {resp.status_code}: {resp.text[:300]}")
        return None

def api_delete(url):
    """Helper for DELETE requests."""
    resp = requests.delete(url, headers=headers)
    if resp.ok or resp.status_code == 204:
        return True
    else:
        print(f"  \u2717 DELETE failed {resp.status_code}: {resp.text[:300]}")
        return False

# Verify access to workspace
ws_info = api_get(f"{PBI_API}/groups/{WORKSPACE_ID}")
if ws_info:
    print(f"\u2713 Connected to workspace: {ws_info.get('name', 'unknown')}")

## Cell 3 - Search all APIs for the ghost item
Queries ALL relevant API surfaces to find anything related to "Anna Asset Delivery Dates". The ghost may appear in one API but not another.

In [ ]:
search_term = REPORT_NAME.lower()
found_items = []

print("=" * 70)
print("SEARCHING ALL API SURFACES")
print("=" * 70)

# --- 1. Fabric Items API (generic) ---
print("\n1\ufe0f\u20e3  Fabric Items API (GET /workspaces/{id}/items)")
items = api_get(f"{FABRIC_API}/workspaces/{WORKSPACE_ID}/items")
if items:
    for item in items.get("value", []):
        if search_term in item.get("displayName", "").lower():
            print(f"   \u27a1 FOUND: {item['displayName']} (type={item['type']}, id={item['id']})")
            found_items.append({"source": "Fabric Items API", **item})
    if not any(search_term in i.get("displayName", "").lower() for i in items.get("value", [])):
        print("   (none found)")

# --- 2. Fabric Paginated Reports API ---
print("\n2\ufe0f\u20e3  Fabric Paginated Reports API (GET /workspaces/{id}/paginatedReports)")
pg_reports = api_get(f"{FABRIC_API}/workspaces/{WORKSPACE_ID}/paginatedReports")
if pg_reports:
    for r in pg_reports.get("value", []):
        if search_term in r.get("displayName", "").lower():
            print(f"   \u27a1 FOUND: {r['displayName']} (id={r['id']})")
            found_items.append({"source": "Fabric PaginatedReports API", **r})
    if not any(search_term in r.get("displayName", "").lower() for r in pg_reports.get("value", [])):
        print("   (none found)")

# --- 3. Power BI Reports API ---
print("\n3\ufe0f\u20e3  Power BI Reports API (GET /groups/{id}/reports)")
pbi_reports = api_get(f"{PBI_API}/groups/{WORKSPACE_ID}/reports")
if pbi_reports:
    for r in pbi_reports.get("value", []):
        if search_term in r.get("name", "").lower():
            print(f"   \u27a1 FOUND: {r['name']} (id={r['id']}, reportType={r.get('reportType', 'N/A')}, datasetId={r.get('datasetId', 'N/A')})")
            found_items.append({"source": "PBI Reports API", "displayName": r["name"], **r})
    if not any(search_term in r.get("name", "").lower() for r in pbi_reports.get("value", [])):
        print("   (none found)")

# --- 4. Power BI Datasets API (the 'model' in the error) ---
print("\n4\ufe0f\u20e3  Power BI Datasets API (GET /groups/{id}/datasets)")
datasets = api_get(f"{PBI_API}/groups/{WORKSPACE_ID}/datasets")
if datasets:
    for ds in datasets.get("value", []):
        if search_term in ds.get("name", "").lower():
            print(f"   \u27a1 FOUND: {ds['name']} (id={ds['id']}, configuredBy={ds.get('configuredBy', 'N/A')})")
            found_items.append({"source": "PBI Datasets API", "displayName": ds["name"], **ds})
    if not any(search_term in ds.get("name", "").lower() for ds in datasets.get("value", [])):
        print("   (none found)")

# --- 5. Power BI Paginated Reports via Reports API (filter by reportType) ---
print("\n5\ufe0f\u20e3  PBI Reports API - Paginated Reports only (reportType=PaginatedReport)")
if pbi_reports:
    for r in pbi_reports.get("value", []):
        if r.get("reportType") == "PaginatedReport" and search_term in r.get("name", "").lower():
            print(f"   \u27a1 FOUND: {r['name']} (id={r['id']}, datasetId={r.get('datasetId', 'N/A')})")
    # Show all paginated reports for reference
    pg_list = [r for r in pbi_reports.get("value", []) if r.get("reportType") == "PaginatedReport"]
    print(f"   Total paginated reports via PBI API: {len(pg_list)}")

# --- 6. Power BI Imports API (check recent imports) ---
print("\n6\ufe0f\u20e3  Power BI Imports API (GET /groups/{id}/imports) - recent imports")
imports = api_get(f"{PBI_API}/groups/{WORKSPACE_ID}/imports")
if imports:
    for imp in imports.get("value", []):
        if search_term in imp.get("name", "").lower():
            print(f"   \u27a1 FOUND: {imp['name']} (id={imp['id']}, importState={imp.get('importState', 'N/A')})")
            # Show associated reports and datasets from this import
            for r in imp.get("reports", []):
                print(f"      Report: {r.get('name', 'N/A')} (id={r.get('id', 'N/A')})")
            for d in imp.get("datasets", []):
                print(f"      Dataset: {d.get('name', 'N/A')} (id={d.get('id', 'N/A')})")
            found_items.append({"source": "PBI Imports API", "displayName": imp["name"], **imp})
    if not any(search_term in imp.get("name", "").lower() for imp in imports.get("value", [])):
        print("   (none found)")

print("\n" + "=" * 70)
print(f"TOTAL MATCHES FOUND: {len(found_items)}")
print("=" * 70)
for item in found_items:
    print(f"  [{item['source']}] {item.get('displayName', 'N/A')} -> id={item.get('id', 'N/A')}")

## Cell 4 - Deep scan for ghost datasets
The error `ReportWithDisplayNameForTheModelExists` means there's a **dataset** (called a 'model' internally) that owns a report name entry matching our report. This cell searches all datasets for the ghost.

In [ ]:
print("=" * 70)
print("DEEP SCAN: Looking for ghost datasets linked to the report name")
print("=" * 70)

ghost_datasets = []
ghost_reports = []

# Check ALL datasets - a previous failed import may have created a dataset
# with the name 'Anna Asset Delivery Dates.rdl' (the RDL filename)
if datasets:
    for ds in datasets.get("value", []):
        ds_name = ds.get("name", "").lower()
        if "anna" in ds_name or "asset delivery" in ds_name:
            ghost_datasets.append(ds)
            print(f"\n\u26a0\ufe0f  GHOST DATASET CANDIDATE:")
            print(f"   Name: {ds['name']}")
            print(f"   ID: {ds['id']}")
            print(f"   Configured By: {ds.get('configuredBy', 'N/A')}")
            print(f"   Created: {ds.get('createdDate', 'N/A')}")
            print(f"   Is Refreshable: {ds.get('isRefreshable', 'N/A')}")
            print(f"   Web URL: {ds.get('webUrl', 'N/A')}")

# Also check reports for ghost entries
if pbi_reports:
    for r in pbi_reports.get("value", []):
        r_name = r.get("name", "").lower()
        if "anna" in r_name or "asset delivery" in r_name:
            ghost_reports.append(r)
            print(f"\n\u26a0\ufe0f  GHOST REPORT CANDIDATE:")
            print(f"   Name: {r['name']}")
            print(f"   ID: {r['id']}")
            print(f"   Report Type: {r.get('reportType', 'N/A')}")
            print(f"   Dataset ID: {r.get('datasetId', 'N/A')}")
            print(f"   Web URL: {r.get('webUrl', 'N/A')}")

if not ghost_datasets and not ghost_reports:
    print("\n\u2139\ufe0f  No ghost items found via standard APIs.")
    print("   The ghost may only be visible to the Import engine internally.")
    print("   Proceed to Cell 5 to try the admin API or the delete-by-name approach.")
else:
    print(f"\n\u2705 Found {len(ghost_datasets)} dataset(s) and {len(ghost_reports)} report(s) to investigate.")
    print("   Proceed to Cell 6 to delete these ghost items.")

## Cell 5 - Try Admin API & RDL-specific dataset name patterns
When importing an RDL, Power BI creates a hidden dataset named `<filename>.rdl`. This cell checks for that exact pattern.

In [ ]:
print("=" * 70)
print("CHECKING FOR HIDDEN RDL-IMPORTED DATASET")
print("=" * 70)

# When PBI Imports API imports an RDL, the dataset gets named '<report_name>.rdl'
rdl_dataset_name = f"{REPORT_NAME}.rdl"
print(f"\nLooking for dataset named: '{rdl_dataset_name}'")

rdl_ghost = None
if datasets:
    for ds in datasets.get("value", []):
        if ds.get("name", "") == rdl_dataset_name:
            rdl_ghost = ds
            print(f"\n\u26a0\ufe0f  FOUND THE GHOST DATASET!")
            print(f"   Name: {ds['name']}")
            print(f"   ID: {ds['id']}")
            print(f"   This is left over from a failed previous import.")
            break

if not rdl_ghost:
    print("   Not found with '.rdl' suffix.")
    # Also check exact name match without .rdl
    print(f"\nLooking for dataset named exactly: '{REPORT_NAME}'")
    if datasets:
        for ds in datasets.get("value", []):
            if ds.get("name", "") == REPORT_NAME:
                rdl_ghost = ds
                print(f"\n\u26a0\ufe0f  FOUND GHOST DATASET (exact name match)!")
                print(f"   Name: {ds['name']}")
                print(f"   ID: {ds['id']}")
                break

# Also list ALL datasets for manual inspection
print("\n" + "-" * 70)
print(f"ALL DATASETS IN WORKSPACE ({len(datasets.get('value', []))} total):")
print("-" * 70)
if datasets:
    for i, ds in enumerate(sorted(datasets.get("value", []), key=lambda x: x.get("name", "")), 1):
        marker = " <<<< SUSPECT" if ("anna" in ds.get("name", "").lower() or "asset delivery" in ds.get("name", "").lower()) else ""
        print(f"   {i:3d}. {ds['name']}{marker}")
        print(f"        id={ds['id']}  type={ds.get('contentProviderType', 'N/A')}")

## Cell 6 - Delete ghost items
Run this ONLY after confirming which items are ghosts from the cells above. This deletes any orphaned dataset and/or report blocking the name.

**Review the output of Cells 3-5 carefully before running this cell.**

In [ ]:
# ===================================================================
# DELETE GHOST ITEMS
# ===================================================================
# This cell collects all candidate ghost items found above and deletes them.
# If the ghost was NOT found by the APIs above, skip to Cell 7 for the
# brute-force approach.
#
# SAFETY: Only items matching "Anna" / "Asset Delivery" are targeted.
# ===================================================================

items_to_delete = []

# Gather ghost datasets
for ds in ghost_datasets:
    items_to_delete.append(("dataset", ds["id"], ds["name"]))
if rdl_ghost and rdl_ghost["id"] not in [d["id"] for _, d_id, _ in items_to_delete if True]:
    items_to_delete.append(("dataset", rdl_ghost["id"], rdl_ghost["name"]))

# Gather ghost reports
for r in ghost_reports:
    items_to_delete.append(("report", r["id"], r["name"]))

if not items_to_delete:
    print("\u26a0\ufe0f  No ghost items identified. Options:")
    print("   a) Re-run Cells 3-5 to search again")
    print("   b) Skip to Cell 7 (manual dataset ID entry)")
    print("   c) Skip to Cell 8 (brute-force: delete ALL items named 'Anna...' and re-import)")
else:
    print(f"Deleting {len(items_to_delete)} ghost item(s):\n")
    for item_type, item_id, item_name in items_to_delete:
        print(f"  Deleting {item_type}: {item_name} (id={item_id})...")
        if item_type == "dataset":
            ok = api_delete(f"{PBI_API}/groups/{WORKSPACE_ID}/datasets/{item_id}")
        elif item_type == "report":
            ok = api_delete(f"{PBI_API}/groups/{WORKSPACE_ID}/reports/{item_id}")
        else:
            ok = api_delete(f"{FABRIC_API}/workspaces/{WORKSPACE_ID}/items/{item_id}")
        
        if ok:
            print(f"   \u2713 Deleted {item_type}: {item_name}")
        else:
            print(f"   \u2717 Failed to delete {item_type}: {item_name}")
    
    print("\n\u2705 Ghost cleanup complete. Proceed to Cell 9 to verify the name is unblocked.")

## Cell 7 - Manual delete (if ghost not found by APIs)
If Cells 3-5 didn't find the ghost, paste the dataset/report ID here manually (e.g., from the Fabric workspace UI or Admin portal).

In [ ]:
# ===================================================================
# MANUAL DELETE - Enter the ID you found in the Fabric workspace UI
# ===================================================================
# Uncomment and fill in one or both IDs if you found the ghost manually.

# GHOST_DATASET_ID = "<paste-dataset-id-here>"  # e.g., from workspace > Datasets
# GHOST_REPORT_ID = "<paste-report-id-here>"    # e.g., from workspace > Reports

# Uncomment the lines below to delete:

# print(f"Deleting dataset {GHOST_DATASET_ID}...")
# api_delete(f"{PBI_API}/groups/{WORKSPACE_ID}/datasets/{GHOST_DATASET_ID}")

# print(f"Deleting report {GHOST_REPORT_ID}...")
# api_delete(f"{PBI_API}/groups/{WORKSPACE_ID}/reports/{GHOST_REPORT_ID}")

print("Cell 7: Manual delete (uncomment lines above to use)")

## Cell 8 - Brute-force cleanup via Fabric Items API
If the ghost is truly invisible to all listing APIs, this cell uses the Fabric generic Items API to find and delete ALL items whose name contains 'Anna'. This is the nuclear option.

In [ ]:
# ===================================================================
# BRUTE-FORCE: List ALL workspace items and delete anything with 'Anna'
# ===================================================================
print("Listing ALL workspace items via Fabric Items API...\n")

all_items = []
continuation_url = f"{FABRIC_API}/workspaces/{WORKSPACE_ID}/items"

while continuation_url:
    resp = requests.get(continuation_url, headers=headers)
    if not resp.ok:
        print(f"\u2717 Failed: {resp.status_code} {resp.text[:200]}")
        break
    data = resp.json()
    all_items.extend(data.get("value", []))
    continuation_url = data.get("continuationUri") or data.get("@odata.nextLink")

print(f"Total items in workspace: {len(all_items)}\n")

anna_items = [item for item in all_items if "anna" in item.get("displayName", "").lower()]

if anna_items:
    print(f"Found {len(anna_items)} item(s) matching 'Anna':")
    for item in anna_items:
        print(f"  - {item['displayName']} (type={item['type']}, id={item['id']})")
    
    print("\nDeleting...")
    for item in anna_items:
        print(f"  Deleting {item['displayName']} ({item['type']})...", end=" ")
        ok = api_delete(f"{FABRIC_API}/workspaces/{WORKSPACE_ID}/items/{item['id']}")
        # Also try PBI-specific endpoints as fallback
        if not ok:
            if item["type"] == "PaginatedReport":
                ok = api_delete(f"{PBI_API}/groups/{WORKSPACE_ID}/reports/{item['id']}")
            elif item["type"] == "Dataset":
                ok = api_delete(f"{PBI_API}/groups/{WORKSPACE_ID}/datasets/{item['id']}")
        print("\u2713" if ok else "\u2717")
    
    print("\n\u2705 Cleanup complete.")
else:
    print("No items matching 'Anna' found even via Fabric Items API.")
    print("\nThe ghost may be a Power BI internal artifact. Try:")
    print("  1. Open the UAT workspace in the Fabric portal")
    print("  2. Search for 'Anna' in the workspace search bar")
    print("  3. Look for ANY item (dataset, report, semantic model) with that name")
    print("  4. Delete it manually, then re-run the pipeline")
    print("\nAlternatively, proceed to Cell 9 to try importing with Overwrite + CreateOrOverwrite.")

## Cell 9 - Verify name is unblocked & test import
After deleting the ghost, this cell verifies the name is free and does a test import with `nameConflict=Abort`.

In [ ]:
import time

print("=" * 70)
print("VERIFICATION: Testing if the name is now unblocked")
print("=" * 70)

# Re-check all surfaces
print("\nChecking Fabric Items API...")
items = api_get(f"{FABRIC_API}/workspaces/{WORKSPACE_ID}/items")
fabric_match = [i for i in (items or {}).get("value", []) if REPORT_NAME.lower() in i.get("displayName", "").lower()]
print(f"  Fabric Items: {len(fabric_match)} match(es)")

print("\nChecking PBI Reports API...")
reports = api_get(f"{PBI_API}/groups/{WORKSPACE_ID}/reports")
pbi_match = [r for r in (reports or {}).get("value", []) if REPORT_NAME.lower() in r.get("name", "").lower()]
print(f"  PBI Reports: {len(pbi_match)} match(es)")

print("\nChecking PBI Datasets API...")
ds = api_get(f"{PBI_API}/groups/{WORKSPACE_ID}/datasets")
ds_match = [d for d in (ds or {}).get("value", []) if REPORT_NAME.lower() in d.get("name", "").lower() or f"{REPORT_NAME}.rdl".lower() in d.get("name", "").lower()]
print(f"  PBI Datasets: {len(ds_match)} match(es)")

if not fabric_match and not pbi_match and not ds_match:
    print("\n\u2705 Name is CLEAR! No items found with this name.")
    print("   You can now re-run the deployment pipeline and it should succeed.")
    print("   Or run Cell 10 below to upload the RDL directly from this notebook.")
else:
    print("\n\u26a0\ufe0f  Items still found! The ghost may not have been fully deleted.")
    print("   Wait 30 seconds and re-run this cell (PBI deletions can be async).")
    for match in fabric_match + pbi_match + ds_match:
        print(f"   - {match.get('displayName', match.get('name', 'N/A'))} (id={match.get('id', 'N/A')})")

## Cell 10 - (Optional) Upload the RDL directly
If you'd like to deploy the Anna report right now without waiting for the pipeline, upload the RDL content here.

In [ ]:
import base64
import time

# ===================================================================
# OPTION A: If the RDL file is accessible from Fabric notebook
# Upload from lakehouse Files or paste the path
# ===================================================================
# RDL_PATH = "/lakehouse/default/Files/Anna Asset Delivery Dates.rdl"

# OPTION B: If you have the .PaginatedReport folder in your Git repo,
# the RDL is at: wsartifacts/PaginatedReports/Anna Asset Delivery Dates.PaginatedReport/Anna Asset Delivery Dates.rdl

# Uncomment ONE of these:
# with open(RDL_PATH, "rb") as f:
#     rdl_bytes = f.read()

# Or paste RDL content as base64 string:
# rdl_bytes = base64.b64decode("<paste-base64-here>")

print("To use this cell:")
print("  1. Upload the .rdl file to the lakehouse Files area")
print("  2. Uncomment the RDL_PATH line and set the path")
print("  3. Uncomment the 'with open(RDL_PATH)' block")
print("  4. Run this cell")
print("")
print("Or simply re-run the deployment pipeline after Cell 9 shows the name is clear.")

In [ ]:
# ===================================================================
# IMPORT via Power BI Imports API (once rdl_bytes is set above)
# ===================================================================
# Uncomment everything below once you have rdl_bytes from the cell above.

# import_url = f"{PBI_API}/groups/{WORKSPACE_ID}/imports?datasetDisplayName={REPORT_NAME}.rdl&nameConflict=Abort"
#
# files = {
#     "file": (f"{REPORT_NAME}.rdl", rdl_bytes, "application/octet-stream")
# }
# import_headers = {
#     "Authorization": f"Bearer {access_token}"
# }
#
# print(f"Importing '{REPORT_NAME}' via Power BI Imports API (nameConflict=Abort)...")
# resp = requests.post(import_url, headers=import_headers, files=files)
#
# if resp.ok:
#     import_data = resp.json()
#     import_id = import_data.get("id")
#     print(f"\u2713 Import initiated (ID: {import_id})")
#
#     # Poll for completion
#     for attempt in range(30):
#         time.sleep(5)
#         status_resp = requests.get(f"{PBI_API}/groups/{WORKSPACE_ID}/imports/{import_id}", headers=headers)
#         if status_resp.ok:
#             status = status_resp.json()
#             state = status.get("importState", "Unknown")
#             print(f"  Attempt {attempt+1}/30: {state}")
#             if state == "Succeeded":
#                 print(f"\n\u2705 Import SUCCEEDED!")
#                 for r in status.get("reports", []):
#                     print(f"  Report: {r.get('name')} (id={r.get('id')})")
#                 break
#             elif state == "Failed":
#                 print(f"\n\u2717 Import FAILED")
#                 print(json.dumps(status, indent=2))
#                 break
# else:
#     print(f"\u2717 Import failed: {resp.status_code}")
#     print(resp.text[:500])

print("Uncomment the code above to perform the import. See instructions in previous cell.")

## Summary

**Root cause:** The error `ReportWithDisplayNameForTheModelExists` with `existingReport: null` means a previous failed import left behind an orphaned dataset/report in Power BI's internal registry. It blocks the name even though the Fabric listing API can't see it.

**Fix steps:**
1. Run Cells 1-5 to identify the ghost item
2. Run Cell 6 or 7 or 8 to delete it
3. Run Cell 9 to verify the name is clear
4. Re-run the deployment pipeline (or use Cell 10/11 to import directly)

**Long-term:** The deploy script should handle this by deleting the ghost dataset before retrying. See the pipeline code fix recommendation.